# Day 074 — Exercise 5: VideoProcessor

**What you'll build:** `VideoProcessor` — full pipeline class binding all four injection functions at construction.

**Why it matters:** The class makes the full pipeline ergonomic: `proc.frames(src, step=5)` reads more clearly than `extract_frames(src, 5, capture_fn=mock)` at every call site.

In [ ]:
import numpy as np
from pathlib import Path

def _make_test_frames(n=10, height=32, width=32):
    frames = []
    for i in range(n):
        frame = np.zeros((height, width, 3), dtype=np.uint8)
        frame[:, :, 0] = int(255 * i / max(n - 1, 1))
        frames.append(frame)
    return frames
_MOCK_META = {
    'fps': 30.0, 'frame_count': 10, 'width': 32, 'height': 32, 'duration_sec': 0.333,
}
_mock_info_fn    = lambda source: dict(_MOCK_META)
_mock_capture_fn = lambda source: _make_test_frames(10)
_mock_writer_fn  = lambda frames, path, fps: (Path(path).write_bytes(b'VIDEO' + bytes(len(frames))), Path(path))[1]
_mock_ffmpeg_fn  = lambda args: {'returncode': 0, 'stdout': '', 'stderr': ''}

def get_video_info(source, info_fn=None):
    if info_fn is not None:
        return info_fn(source)
    import cv2
    cap = cv2.VideoCapture(str(source))
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {source}')
    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width       = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = round(frame_count / fps, 3) if fps > 0 else 0.0
    return {'fps': fps, 'frame_count': frame_count,
            'width': width, 'height': height, 'duration_sec': duration}

def extract_frames(source, step=1, max_frames=None, capture_fn=None):
    if capture_fn is not None:
        all_frames = capture_fn(source)
        stepped    = all_frames[::step]
        return stepped[:max_frames] if max_frames is not None else stepped
    import cv2
    cap = cv2.VideoCapture(str(source)); frames = []; idx = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if idx % step == 0:
            frames.append(frame)
            if max_frames is not None and len(frames) >= max_frames: break
        idx += 1
    cap.release(); return frames

def frames_to_video(frames, output_path, fps=30.0, fourcc='mp4v', writer_fn=None):
    if writer_fn is not None:
        return writer_fn(frames, output_path, fps)
    import cv2; frames = list(frames)
    if not frames: raise ValueError('frames list is empty')
    h, w = frames[0].shape[:2]; code = cv2.VideoWriter_fourcc(*fourcc)
    writer = cv2.VideoWriter(str(Path(output_path)), code, fps, (w, h))
    for f in frames: writer.write(f)
    writer.release(); return Path(output_path)

def run_ffmpeg(args, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(args)
    import subprocess
    r = subprocess.run(['ffmpeg', '-y'] + list(args), capture_output=True, text=True)
    return {'returncode': r.returncode, 'stdout': r.stdout, 'stderr': r.stderr}


## Task

Implement `VideoProcessor`:

- `__init__`: store `info_fn`, `capture_fn`, `writer_fn`, `ffmpeg_fn` as `self._xxx_fn`
- `info(source)`: `return get_video_info(source, info_fn=self._info_fn)`
- `frames(source, step=1, max_frames=None)`: delegate to `extract_frames` with `capture_fn=self._capture_fn`
- `to_video(frames, output_path, fps=30.0, fourcc='mp4v')`: delegate to `frames_to_video` with `writer_fn=self._writer_fn`
- `run_ffmpeg(args)`: delegate to `run_ffmpeg` with `ffmpeg_fn=self._ffmpeg_fn`

## Your Implementation

In [ ]:
class VideoProcessor:
    """Process video files using OpenCV and FFmpeg.

    Inject fn parameters for testing without video files or FFmpeg.
    """

    def __init__(self, info_fn=None, capture_fn=None,
                 writer_fn=None, ffmpeg_fn=None) -> None:
        raise NotImplementedError

    def info(self, source) -> dict:
        """Return video metadata dict."""
        raise NotImplementedError

    def frames(self, source, step: int = 1, max_frames=None) -> list:
        """Extract frames as a list of numpy arrays."""
        raise NotImplementedError

    def to_video(self, frames: list, output_path,
                 fps: float = 30.0, fourcc: str = 'mp4v'):
        """Write frames to a video file. Returns Path."""
        raise NotImplementedError

    def run_ffmpeg(self, args: list) -> dict:
        """Run an FFmpeg command. Returns result dict."""
        raise NotImplementedError


In [ ]:
class VideoProcessor:
    def __init__(self, info_fn=None, capture_fn=None,
                 writer_fn=None, ffmpeg_fn=None):
        self._info_fn    = info_fn
        self._capture_fn = capture_fn
        self._writer_fn  = writer_fn
        self._ffmpeg_fn  = ffmpeg_fn

    def info(self, source):
        return get_video_info(source, info_fn=self._info_fn)

    def frames(self, source, step=1, max_frames=None):
        return extract_frames(source, step=step,
                              max_frames=max_frames,
                              capture_fn=self._capture_fn)

    def to_video(self, frames, output_path, fps=30.0, fourcc='mp4v'):
        return frames_to_video(frames, output_path, fps=fps,
                               fourcc=fourcc, writer_fn=self._writer_fn)

    def run_ffmpeg(self, args):
        return run_ffmpeg(args, ffmpeg_fn=self._ffmpeg_fn)


## Automated checks

In [ ]:

import tempfile
score, total = 0, 5
try:
    proc = VideoProcessor(
        info_fn=_mock_info_fn,
        capture_fn=_mock_capture_fn,
        writer_fn=_mock_writer_fn,
        ffmpeg_fn=_mock_ffmpeg_fn,
    )

    # info delegates to get_video_info
    meta = proc.info('video.mp4')
    assert isinstance(meta, dict) and 'fps' in meta and 'frame_count' in meta
    score += 1; print("✅ info() returns metadata dict")

    # frames delegates to extract_frames
    frames = proc.frames('video.mp4', step=2, max_frames=4)
    assert isinstance(frames, list) and len(frames) <= 4
    assert frames[0].shape == (32, 32, 3)
    score += 1; print("✅ frames() returns correct list of numpy arrays")

    # to_video delegates to frames_to_video
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        tmp = f.name
    out = proc.to_video(_make_test_frames(5), tmp)
    assert isinstance(out, Path) and out.exists()
    score += 1; print("✅ to_video() returns existing Path")

    # run_ffmpeg delegates correctly
    res = proc.run_ffmpeg(['-i', 'in.mp4', 'out.avi'])
    assert res.get('returncode') == 0
    score += 1; print("✅ run_ffmpeg() returns result dict with returncode 0")

    # injection fns bound at construction (not passed per call)
    captured = {}
    def _cap_capture(src): captured['called'] = True; return _make_test_frames(5)
    proc2 = VideoProcessor(capture_fn=_cap_capture)
    proc2.frames('x.mp4')
    assert captured.get('called') is True
    score += 1; print("✅ injection fn bound at construction, used on each call")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class VideoProcessor:
    def __init__(self, info_fn=None, capture_fn=None,
                 writer_fn=None, ffmpeg_fn=None):
        self._info_fn    = info_fn
        self._capture_fn = capture_fn
        self._writer_fn  = writer_fn
        self._ffmpeg_fn  = ffmpeg_fn

    def info(self, source):
        return get_video_info(source, info_fn=self._info_fn)

    def frames(self, source, step=1, max_frames=None):
        return extract_frames(source, step=step,
                              max_frames=max_frames,
                              capture_fn=self._capture_fn)

    def to_video(self, frames, output_path, fps=30.0, fourcc='mp4v'):
        return frames_to_video(frames, output_path, fps=fps,
                               fourcc=fourcc, writer_fn=self._writer_fn)

    def run_ffmpeg(self, args):
        return run_ffmpeg(args, ffmpeg_fn=self._ffmpeg_fn)
```

**Why store as `self._xxx_fn`** (underscore prefix)? Underscore signals implementation detail — callers use `.info()`, `.frames()` etc, not the raw injection functions. Same convention as `_describe_fn` in Day 67's `VisionAnalyzer`.

</details>